<a href="https://colab.research.google.com/github/Razae786/ANN-Lab-FA23-BAI-064/blob/main/NLP_Lab_05_Solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Lab 05: N-gram Language Models

This notebook uses only the Python standard library.

Topics:
- Tokenization
- Unigrams, bigrams, trigrams
- N-gram counting
- MLE probabilities
- Zero-probability problem
- Add-one and Add-k smoothing
- Perplexity
- Model comparison

In [1]:
import math
from collections import Counter
from itertools import permutations

# Data
train = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
    "the dog ate the bone"
]

test = [
    "the cat sat on the rug",
    "the fish sat on the mat"
]

print("Training sentences:")
for s in train:
    print(s)

print("\nTest sentences:")
for s in test:
    print(s)

Training sentences:
the cat sat on the mat
the cat ate the fish
the dog sat on the rug
the dog ate the bone

Test sentences:
the cat sat on the rug
the fish sat on the mat


## Task 1: Tokenization

In [2]:
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]


print("Tokenized training data:")
for s in train:
    print(s, "->", tokenize(s))

print("\nTokenized test data:")
for s in test:
    print(s, "->", tokenize(s))


train_tokens_with_markers = sum(len(tokenize(s)) for s in train)
train_tokens_without_markers = sum(len(s.lower().split()) for s in train)

print("\nTotal training tokens with markers:", train_tokens_with_markers)
print("Total training tokens without markers:", train_tokens_without_markers)


def build_vocab(sentences):
    vocab = set()
    for sentence in sentences:
        vocab.update(sentence.lower().split())

    # Add </s>, but not <s>
    vocab.add("</s>")
    return vocab


vocab = build_vocab(train)
V = len(vocab)

print("\nVocabulary size V:", V)
print("Sorted vocabulary:", sorted(vocab))

Tokenized training data:
the cat sat on the mat -> ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
the cat ate the fish -> ['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
the dog sat on the rug -> ['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
the dog ate the bone -> ['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']

Tokenized test data:
the cat sat on the rug -> ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
the fish sat on the mat -> ['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']

Total training tokens with markers: 30
Total training tokens without markers: 22

Vocabulary size V: 11
Sorted vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


## Task 2: Generating N-grams

In [3]:
def get_ngrams(tokens, n):
    if n <= 0:
        return []
    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]


first_tokens = tokenize(train[0])

print("First sentence tokens:")
print(first_tokens)

print("\nUnigrams:")
print(get_ngrams(first_tokens, 1))

print("\nBigrams:")
print(get_ngrams(first_tokens, 2))

print("\nTrigrams:")
print(get_ngrams(first_tokens, 3))


print("\nChecking m - n + 1 rule:")
for sentence in train:
    tokens = tokenize(sentence)
    m = len(tokens)

    for n in [1, 2, 3]:
        expected = m - n + 1
        actual = len(get_ngrams(tokens, n))

        assert actual == expected
        print(
            f"m={m}, n={n}, expected={expected}, actual={actual} -> OK"
        )


print("\nN larger than sentence length:")
short_tokens = ["<s>", "hello", "</s>"]

print("Tokens:", short_tokens)
print("n=5:", get_ngrams(short_tokens, 5))

First sentence tokens:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']

Unigrams:
[('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]

Bigrams:
[('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]

Trigrams:
[('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]

Checking m - n + 1 rule:
m=8, n=1, expected=8, actual=8 -> OK
m=8, n=2, expected=7, actual=7 -> OK
m=8, n=3, expected=6, actual=6 -> OK
m=7, n=1, expected=7, actual=7 -> OK
m=7, n=2, expected=6, actual=6 -> OK
m=7, n=3, expected=5, actual=5 -> OK
m=8, n=1, expected=8, actual=8 -> OK
m=8, n=2, expected=7, actual=7 -> OK
m=8, n=3, expected=6, actual=6 -> OK
m=7, n=1, expected=7, actual=7 -> OK
m=7, n=2, expected=6, actual=6 -> OK
m=7, n=3, expected=5, actual=5 -> OK

N larger than sentence length:
Tokens: ['<s>', 'hello', '</s>']
n=5: []


## Task 3: Counting N-grams

In [4]:
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for sentence in train:
    tokens = tokenize(sentence)

    unigram_counts.update(get_ngrams(tokens, 1))
    bigram_counts.update(get_ngrams(tokens, 2))
    trigram_counts.update(get_ngrams(tokens, 3))


def print_counter_table(counter, title):
    print("\n" + title)
    print("-" * len(title))

    for ngram, count in sorted(
        counter.items(),
        key=lambda x: (-x[1], x[0])
    ):
        print(f"{ngram}: {count}")


print_counter_table(unigram_counts, "Unigram Counts")
print_counter_table(bigram_counts, "Bigram Counts")
print_counter_table(trigram_counts, "Trigram Counts")


print("\n5 most common bigrams:")
for bigram, count in bigram_counts.most_common(5):
    print(bigram, ":", count)


print("\nNumber of distinct unigrams:", len(unigram_counts))
print("Number of distinct bigrams:", len(bigram_counts))
print("Number of distinct trigrams:", len(trigram_counts))


def count_of(ngram):
    ngram = tuple(ngram)

    if len(ngram) == 1:
        return unigram_counts[ngram]
    elif len(ngram) == 2:
        return bigram_counts[ngram]
    elif len(ngram) == 3:
        return trigram_counts[ngram]
    else:
        return 0


print("\ncount_of tests:")
print("Seen bigram ('the', 'cat'):", count_of(("the", "cat")))
print("Unseen bigram ('cat', 'dog'):", count_of(("cat", "dog")))


Unigram Counts
--------------
('the',): 8
('</s>',): 4
('<s>',): 4
('ate',): 2
('cat',): 2
('dog',): 2
('on',): 2
('sat',): 2
('bone',): 1
('fish',): 1
('mat',): 1
('rug',): 1

Bigram Counts
-------------
('<s>', 'the'): 4
('ate', 'the'): 2
('on', 'the'): 2
('sat', 'on'): 2
('the', 'cat'): 2
('the', 'dog'): 2
('bone', '</s>'): 1
('cat', 'ate'): 1
('cat', 'sat'): 1
('dog', 'ate'): 1
('dog', 'sat'): 1
('fish', '</s>'): 1
('mat', '</s>'): 1
('rug', '</s>'): 1
('the', 'bone'): 1
('the', 'fish'): 1
('the', 'mat'): 1
('the', 'rug'): 1

Trigram Counts
--------------
('<s>', 'the', 'cat'): 2
('<s>', 'the', 'dog'): 2
('sat', 'on', 'the'): 2
('ate', 'the', 'bone'): 1
('ate', 'the', 'fish'): 1
('cat', 'ate', 'the'): 1
('cat', 'sat', 'on'): 1
('dog', 'ate', 'the'): 1
('dog', 'sat', 'on'): 1
('on', 'the', 'mat'): 1
('on', 'the', 'rug'): 1
('the', 'bone', '</s>'): 1
('the', 'cat', 'ate'): 1
('the', 'cat', 'sat'): 1
('the', 'dog', 'ate'): 1
('the', 'dog', 'sat'): 1
('the', 'fish', '</s>'): 1
('the',

## Task 4: Unigram Model

In [5]:
N = sum(unigram_counts.values()) - unigram_counts[("<s>",)]

# Probability vocabulary excludes <s>.
def unigram_prob(w):
    return unigram_counts[(w,)] / N


print("Unigram probabilities:")
unigram_total = 0

for word in sorted(vocab):
    p = unigram_prob(word)
    unigram_total += p
    print(f"{word:10} {p:.4f}")

assert math.isclose(unigram_total, 1.0, rel_tol=1e-9)
print("Sum of unigram probabilities =", round(unigram_total, 4))


def unigram_sentence_prob(sentence):
    probability = 1.0

    for word in tokenize(sentence):
        if word == "<s>":
            continue
        probability *= unigram_prob(word)

    return probability


print("\nUnigram sentence probabilities:")
for sentence in test:
    print(f"{sentence}: {unigram_sentence_prob(sentence):.8f}")


# Different word orders can have the same unigram probability.
words = test[0].lower().split()

print("\nPermutation example:")
p1 = unigram_sentence_prob(" ".join(words))

found_different = False

for perm in permutations(words):
    sentence = " ".join(perm)
    p = unigram_sentence_prob(sentence)

    if sentence != " ".join(words) and math.isclose(p, p1):
        print("Original:", " ".join(words))
        print("Different order:", sentence)
        print("Same unigram probability:", round(p, 8))
        found_different = True
        break

assert found_different

Unigram probabilities:
</s>       0.1538
ate        0.0769
bone       0.0385
cat        0.0769
dog        0.0769
fish       0.0385
mat        0.0385
on         0.0769
rug        0.0385
sat        0.0769
the        0.3077
Sum of unigram probabilities = 1.0

Unigram sentence probabilities:
the cat sat on the rug: 0.00000025
the fish sat on the mat: 0.00000013

Permutation example:
Original: the cat sat on the rug
Different order: the cat sat on rug the
Same unigram probability: 2.5e-07


## Task 5: Bigram Model, MLE

In [6]:
def bigram_prob(prev, w):
    denominator = unigram_counts[(prev,)]

    if denominator == 0:
        return 0.0

    return bigram_counts[(prev, w)] / denominator


def next_word_distribution(prev):
    denominator = unigram_counts[(prev,)]

    if denominator == 0:
        return {}

    distribution = {}

    for word in sorted(vocab):
        p = bigram_prob(prev, word)
        if p > 0:
            distribution[word] = p

    return distribution


print("Next-word distribution after 'the':")
the_distribution = next_word_distribution("the")

for word, p in the_distribution.items():
    print(f"P({word} | the) = {p:.4f}")

assert math.isclose(sum(the_distribution.values()), 1.0, rel_tol=1e-9)


def bigram_sentence_prob(sentence):
    probability = 1.0
    tokens = tokenize(sentence)

    for prev, word in get_ngrams(tokens, 2):
        probability *= bigram_prob(prev, word)

    return probability


print("\nBigram sentence probabilities:")
for sentence in test:
    print(f"{sentence}: {bigram_sentence_prob(sentence):.8f}")


def predict_next(prev):
    distribution = next_word_distribution(prev)

    if not distribution:
        return None

    return min(
        distribution,
        key=lambda word: (-distribution[word], word)
    )


print("\nNext-word predictions:")
for word in ["the", "cat", "dog", "sat", "ate"]:
    print(f"After '{word}' -> '{predict_next(word)}'")

Next-word distribution after 'the':
P(bone | the) = 0.1250
P(cat | the) = 0.2500
P(dog | the) = 0.2500
P(fish | the) = 0.1250
P(mat | the) = 0.1250
P(rug | the) = 0.1250

Bigram sentence probabilities:
the cat sat on the rug: 0.01562500
the fish sat on the mat: 0.00000000

Next-word predictions:
After 'the' -> 'cat'
After 'cat' -> 'ate'
After 'dog' -> 'ate'
After 'sat' -> 'on'
After 'ate' -> 'the'


## Task 6: Trigram Model

In [7]:
def trigram_prob(w1, w2, w3):
    denominator = bigram_counts[(w1, w2)]

    if denominator == 0:
        return 0.0

    return trigram_counts[(w1, w2, w3)] / denominator


def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)

    if len(tokens) < 2:
        return 0.0

    # First word is predicted using the bigram model.
    probability = bigram_prob(tokens[0], tokens[1])

    # Remaining words use the trigram model.
    for w1, w2, w3 in get_ngrams(tokens, 3):
        probability *= trigram_prob(w1, w2, w3)

    return probability


print("Trigram sentence probabilities:")
for sentence in test:
    print(f"{sentence}: {trigram_sentence_prob(sentence):.8f}")


print("\nN-gram counts:")
print("Type       Distinct")
print("Unigram    ", len(unigram_counts))
print("Bigram     ", len(bigram_counts))
print("Trigram    ", len(trigram_counts))


def unseen_ngrams(sentence, n, counter):
    tokens = tokenize(sentence)
    return [
        ngram
        for ngram in get_ngrams(tokens, n)
        if counter[ngram] == 0
    ]


print("\nUnseen bigrams and trigrams:")

for sentence in test:
    unseen_bi = unseen_ngrams(sentence, 2, bigram_counts)
    unseen_tri = unseen_ngrams(sentence, 3, trigram_counts)

    print("\nSentence:", sentence)
    print("Unseen bigrams:", unseen_bi)
    print("Unseen trigrams:", unseen_tri)

    if len(unseen_bi) > len(unseen_tri):
        print("Model with more unseen N-grams: Bigram")
    elif len(unseen_tri) > len(unseen_bi):
        print("Model with more unseen N-grams: Trigram")
    else:
        print("Both models have the same number of unseen N-grams.")

Trigram sentence probabilities:
the cat sat on the rug: 0.12500000
the fish sat on the mat: 0.00000000

N-gram counts:
Type       Distinct
Unigram     12
Bigram      18
Trigram     19

Unseen bigrams and trigrams:

Sentence: the cat sat on the rug
Unseen bigrams: []
Unseen trigrams: []
Both models have the same number of unseen N-grams.

Sentence: the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
Model with more unseen N-grams: Trigram


## Task 7: Zero-Probability Problem

In [8]:
def unseen_bigrams(sentence):
    return [
        bigram
        for bigram in get_ngrams(tokenize(sentence), 2)
        if bigram_counts[bigram] == 0
    ]


for sentence in test:
    print("\nSentence:", sentence)

    unseen = unseen_bigrams(sentence)

    for bigram in unseen:
        p = bigram_prob(bigram[0], bigram[1])
        print(f"{bigram}: probability = {p:.4f}")

    print("Number of unseen bigrams:", len(unseen))


# Count possible word pairs without markers.
words_only = sorted(set(
    word
    for sentence in train
    for word in sentence.lower().split()
))

total_possible_pairs = len(words_only) ** 2
seen_pairs = sum(
    1
    for w1 in words_only
    for w2 in words_only
    if bigram_counts[(w1, w2)] > 0
)

unseen_pairs = total_possible_pairs - seen_pairs
unseen_percentage = 100 * unseen_pairs / total_possible_pairs

print("\nPossible word pairs:", total_possible_pairs)
print("Seen word pairs:", seen_pairs)
print("Unseen word pairs:", unseen_pairs)
print(f"Unseen percentage: {unseen_percentage:.2f}%")


def safe_log(p):
    if p == 0:
        return float("-inf")
    return math.log(p)


second_test = test[1]
second_log_prob = 0.0

for prev, word in get_ngrams(tokenize(second_test), 2):
    second_log_prob += safe_log(bigram_prob(prev, word))

print("\nLog probability of second test sentence:")
print(second_log_prob)


Sentence: the cat sat on the rug
Number of unseen bigrams: 0

Sentence: the fish sat on the mat
('fish', 'sat'): probability = 0.0000
Number of unseen bigrams: 1

Possible word pairs: 100
Seen word pairs: 13
Unseen word pairs: 87
Unseen percentage: 87.00%

Log probability of second test sentence:
-inf


## Task 8: Smoothing

In [10]:
def bigram_prob_addk(prev, w, k):
    return (
        bigram_counts[(prev, w)] + k
    ) / (
        unigram_counts[(prev,)] + k * V
    )


def bigram_prob_addone(prev, w):
    return bigram_prob_addk(prev, w, 1)


print("Probability comparison:")
print("Word pair          MLE       Add-One")

for prev, word in [("fish", "sat"), ("the", "cat")]:
    mle = bigram_prob(prev, word)
    addone = bigram_prob_addone(prev, word)

    print(
        f"P({word} | {prev})"
        f"{mle:12.4f}{addone:12.4f}"
    )


print("\nChecking Add-One distributions:")

# We exclude </s> as a preceding word since it is the end-of-sentence marker and does not predict a next word.
for prev in vocab:
    if prev == "</s>":
        continue

    probabilities = [
        bigram_prob_addone(prev, word)
        for word in vocab
    ]

    total = sum(probabilities)

    assert math.isclose(total, 1.0, rel_tol=1e-9)

print("All smoothed distributions sum to 1.")


print("\nAdd-k values for P(sat | fish):")
print("k       Probability")

for k in [1, 0.5, 0.1, 0.01]:
    p = bigram_prob_addk("fish", "sat", k)
    print(f"{k:<7} {p:.4f}")

Probability comparison:
Word pair          MLE       Add-One
P(sat | fish)      0.0000      0.0833
P(cat | the)      0.2500      0.1579

Checking Add-One distributions:
All smoothed distributions sum to 1.

Add-k values for P(sat | fish):
k       Probability
1       0.0833
0.5     0.0769
0.1     0.0476
0.01    0.0090


## Task 9: Perplexity and Model Comparison

In [11]:
def perplexity(sentences, prob_fn):
    log_sum = 0.0
    count = 0

    for sentence in sentences:
        tokens = tokenize(sentence)

        for prev, word in get_ngrams(tokens, 2):
            p = prob_fn(prev, word)

            if p == 0:
                return float("inf")

            log_sum += math.log(p)
            count += 1

    return math.exp(-log_sum / count)


def unigram_prob_addone(w):
    return (
        unigram_counts[(w,)] + 1
    ) / (
        N + V
    )


bigram_mle_pp = perplexity(test, bigram_prob)
bigram_addone_pp = perplexity(test, bigram_prob_addone)
unigram_addone_pp = perplexity(
    test,
    lambda prev, w: unigram_prob_addone(w)
)

print("Perplexity comparison:")
print("Model                    Perplexity")
print(
    f"Bigram MLE              {bigram_mle_pp:.4f}"
    if math.isfinite(bigram_mle_pp)
    else "Bigram MLE              infinity"
)
print(f"Bigram Add-One           {bigram_addone_pp:.4f}")
print(f"Unigram Add-One          {unigram_addone_pp:.4f}")


print("\nAdd-k perplexity:")
print("k       Perplexity")

k_results = {}

for k in [1, 0.5, 0.1, 0.01]:
    pp = perplexity(
        test,
        lambda prev, w, k=k: bigram_prob_addk(prev, w, k)
    )

    k_results[k] = pp
    print(f"{k:<7} {pp:.4f}")

best_k = min(k_results, key=k_results.get)

print(f"\nBest k on test set: {best_k}")
print(f"Best test perplexity: {k_results[best_k]:.4f}")

Perplexity comparison:
Model                    Perplexity
Bigram MLE              infinity
Bigram Add-One           5.7969
Unigram Add-One          9.1370

Add-k perplexity:
k       Perplexity
1       5.7969
0.5     4.5703
0.1     2.9559
0.01    2.6272

Best k on test set: 0.01
Best test perplexity: 2.6272


## Task 9.4: Validation-based k Selection

In [12]:
# Train on only the first three sentences.
validation_train = train[:3]
validation_sentence = train[3]
validation_test = test

# Build counts from the first three sentences.
val_unigram_counts = Counter()
val_bigram_counts = Counter()

for sentence in validation_train:
    tokens = tokenize(sentence)

    val_unigram_counts.update(get_ngrams(tokens, 1))
    val_bigram_counts.update(get_ngrams(tokens, 2))

# V is built from all four training sentences, as required.
validation_V = V

def validation_bigram_addk(prev, w, k):
    return (
        val_bigram_counts[(prev, w)] + k
    ) / (
        val_unigram_counts[(prev,)] + k * validation_V
    )


def sentence_perplexity(sentences, prob_fn):
    log_sum = 0.0
    count = 0

    for sentence in sentences:
        for prev, word in get_ngrams(tokenize(sentence), 2):
            p = prob_fn(prev, word)

            if p == 0:
                return float("inf")

            log_sum += math.log(p)
            count += 1

    return math.exp(-log_sum / count)


validation_results = {}

print("Validation perplexity:")
print("k       Validation PP")

for k in [1, 0.5, 0.1, 0.01]:
    pp = sentence_perplexity(
        [validation_sentence],
        lambda prev, w, k=k: validation_bigram_addk(prev, w, k)
    )

    validation_results[k] = pp
    print(f"{k:<7} {pp:.4f}")

best_validation_k = min(
    validation_results,
    key=validation_results.get
)

print("\nValidation sentence:", validation_sentence)
print("Best k selected from validation:", best_validation_k)
print(
    "Validation perplexity:",
    round(validation_results[best_validation_k], 4)
)


# Report test perplexity using the k selected from validation.
test_pp_using_validation_k = perplexity(
    validation_test,
    lambda prev, w: bigram_prob_addk(
        prev, w, best_validation_k
    )
)

print("\nTest perplexity using validation-selected k:")
print(round(test_pp_using_validation_k, 4))

print("\nComparison:")
print(
    "Best k chosen directly on test set:",
    best_k
)
print(
    "Test perplexity with test-selected k:",
    round(k_results[best_k], 4)
)
print(
    "Best k chosen using validation data:",
    best_validation_k
)
print(
    "Test perplexity with validation-selected k:",
    round(test_pp_using_validation_k, 4)
)

Validation perplexity:
k       Validation PP
1       8.5857
0.5     8.0163
0.1     8.0260
0.01    13.1304

Validation sentence: the dog ate the bone
Best k selected from validation: 0.5
Validation perplexity: 8.0163

Test perplexity using validation-selected k:
4.5703

Comparison:
Best k chosen directly on test set: 0.01
Test perplexity with test-selected k: 2.6272
Best k chosen using validation data: 0.5
Test perplexity with validation-selected k: 4.5703
